# Example: Symmetry-reduced Pulse Design Tool

---

This notebook demonstrates how to use `SymmetryReducedSolver` with the **FreeGSNKE Pulse Design Tool** (FPDT) on an Anamak-like tokamak, evolving the coupled plasma/metal system strictly in the even-in-$Z$ subspace.

This should enable large timesteps to be taken and rapid prototyping of plasma scenarios.

In this notebook, we:
1. Prepare an exactly up-down symmetric machine description.
2. Construct a symmetric initial Grad-Shafranov equilibrium.
3. Inspect the even and odd passive structure modes.
4. Run a closed-loop FPDT simulation using `SymmetryReducedSolver` in place of `nl_solver`.

Note that this is a very basic example on a tokamak with few coils, so the control demonstrated here is limited.

### 1. Build the analytic source machine

We place active coils and passive structures in reflected pairs about $Z = 0$: five shaping pairs, one symmetric central solenoid, and one odd anti-series vertical-control circuit (included to demonstrate explicit exclusion during preparation). The fifth shaping pair sits close to the X-points: the other four pairs act almost identically on the X-point radius $R_x$ and the midplane radii, so moving $R_x$ on its own would need very large, nearly cancelling currents. The limiter radius is chosen to avoid landing directly on grid nodes.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np

from freegsnke import GSstaticsolver, build_machine, equilibrium_update
from freegsnke.inverse import Inverse_optimizer
from freegsnke.jtor_update import Lao85
from freegsnke.up_down_symmetry import prepare_up_down_symmetric_machine

In [ ]:
# build the symmetric machine data
def active_element(r, z, multiplier=100, polarity=1):
    """Return one rectangular active-coil filament bundle."""
    return {
        "R": [r],
        "Z": [z],
        "dR": 0.06,
        "dZ": 0.06,
        "resistivity": 1.55e-8,
        "polarity": polarity,
        "multiplier": multiplier,
    }


def analytic_source_machine():
    """Build an approximately machine-shaped, exactly paired toy description."""
    major_radius = 1.0
    active = {
        "solenoid": {
            **active_element(0.28, 0.0, multiplier=20),
            "R": [0.28] * 4,
            "Z": [-0.55, -0.18, 0.18, 0.55],
        }
    }

    upper_angles = (np.pi / 4, 3 * np.pi / 4, np.pi / 8, 7 * np.pi / 8)
    for index, angle in enumerate(upper_angles, start=1):
        r = major_radius + 0.70 * np.cos(angle)
        z = 0.70 * np.sin(angle)
        active[f"PF_{index}_upper"] = active_element(r, z)
        active[f"PF_{index}_lower"] = active_element(r, -z)

    # fifth shaping pair, close to the X-points, for independent control of the X-point position
    active["PF_5_upper"] = active_element(0.75, 0.60)
    active["PF_5_lower"] = active_element(0.75, -0.60)

    active["vertical_control"] = {
        "upper": active_element(1.68, 0.28),
        "lower": active_element(1.68, -0.28, polarity=-1),
    }

    passive = []
    for index, angle in enumerate(np.linspace(np.pi / 20, 19 * np.pi / 20, 10)):
        r = major_radius + 0.50 * np.cos(angle)
        z = 0.50 * np.sin(angle)
        half_width = 0.025
        for side, z_centre in (("upper", z), ("lower", -z)):
            passive.append(
                {
                    "name": f"shell_{index}_{side}",
                    "element": f"shell_{index}",
                    "R": np.array(
                        [
                            r - half_width,
                            r + half_width,
                            r + half_width,
                            r - half_width,
                        ]
                    ),
                    "Z": np.array(
                        [
                            z_centre - half_width,
                            z_centre - half_width,
                            z_centre + half_width,
                            z_centre + half_width,
                        ]
                    ),
                    "resistivity": 5.5e-7,
                }
            )

    theta = np.linspace(0.0, 2 * np.pi, 96, endpoint=True)
    limiter = [
        {
            "R": float(major_radius + 0.44 * np.cos(angle)),
            "Z": float(0.44 * np.sin(angle)),
        }
        for angle in theta
    ]
    return active, passive, limiter

### Build the symmetrised machine

Let's build the symmetric machine (excluding the vertical control coil).

In [ ]:
# machine data
active_source, passive_source, limiter_source = analytic_source_machine()

# symmetric machine data
symmetric_machine = prepare_up_down_symmetric_machine(
    active_source,
    passive_source,
    limiter_data=limiter_source,
    wall_data=limiter_source,
    exclude_odd_active=True,
)

In [ ]:
# build machine
tokamak = build_machine.tokamak(
    active_coils_data=symmetric_machine.even_active_coils_data,
    passive_coils_data=symmetric_machine.passive_coils_data,
    limiter_data=symmetric_machine.limiter_data,
    wall_data=symmetric_machine.wall_data,
)

In [ ]:
# plot machine
fig, ax = plt.subplots(figsize=(6, 6), dpi=80)
tokamak.plot(axis=ax, show=False)
ax.plot(tokamak.limiter.R, tokamak.limiter.Z, "k-", linewidth=1.2)
ax.set( xlabel="Major radius, R [m]", ylabel="Height, Z [m]", aspect="equal")
ax.grid(zorder=0, alpha=0.75)
ax.set_aspect('equal')
plt.show()


### 2. Construct a symmetric initial equilibrium

Let's use the inverse solver to build an initial equilibrium. 

In [ ]:
# initialise eq
eq = equilibrium_update.Equilibrium(
    tokamak=tokamak,
    Rmin=0.1,
    Rmax=1.9,
    Zmin=-0.8,
    Zmax=0.8,
    nx=65,
    ny=65,
)

# initialise profiles
profiles = Lao85(
    eq=eq,
    Ip=2.0e5,
    fvac=0.5,
    alpha=[58213.6],
    beta=[0.582136],
)

# static solver
static_solver = GSstaticsolver.NKGSsolver(eq)

In [ ]:
# constraints for inverse solver
tokamak.set_coil_current("solenoid", 3000.0)
tokamak["solenoid"].control = False

# the extra shaping pair carries no current in the initial equilibrium (so the equilibrium is
# unchanged); it is only used by the shape controllers
tokamak.set_coil_current("PF_5", 0.0)
tokamak["PF_5"].control = False

constrain = Inverse_optimizer(
    null_points=[[0.8, 0.8], [0.35, -0.35]],
    isoflux_set=np.array([[[0.8, 0.8, 1.22], [0.35, -0.35, 0.0]]]),
    weight_isoflux=1.0,
    weight_nulls=0.8,
)


In [ ]:
# the numerical operators need exactly up-down symmetric resistances and inductances, so these calls enforce this
eq._vgreen = symmetric_machine.symmetrise_even_machine_greens(eq._vgreen)
tokamak.coil_resist = symmetric_machine.symmetrise_even_machine_resistances(
    tokamak.coil_resist
)
tokamak.coil_self_ind = symmetric_machine.symmetrise_even_machine_square_operator(
    tokamak.coil_self_ind
)

In [ ]:
# solve and plot
static_solver.inverse_solve(
    eq=eq,
    profiles=profiles,
    constrain=constrain,
    target_relative_tolerance=1e-5,
    target_relative_psit_update=1e-3,
    l2_reg=np.full(4, 1e-8),
    force_up_down_symmetric=True,
    verbose=False,
)


We should have a nice up-down symmetric equilibrium in the symmetric machine!

In [ ]:
# plot the equilibrium
fig1, ax1 = plt.subplots(1, 1, figsize=(7, 15), dpi=80)
ax1.grid(zorder=0, alpha=0.75)
ax1.set_aspect('equal')
eq.tokamak.plot(axis=ax1,show=False)                                                          # plots the active coils and passive structures
ax1.fill(tokamak.wall.R, tokamak.wall.Z, color='k', linewidth=1.2, facecolor='w', zorder=0)   # plots the limiter
eq.plot(axis=ax1,show=False)                                                                  # plots the equilibrium
constrain.plot(axis=ax1, show=False)                                                          # plots the constraints

If needed we can do some checks to ensure everything is fully up-down symmetric (i.e. machine operators commute with the metal-current reflection, the limiter mask is reflection-invariant, and the initial plasma fields and metal currents are even to round-off). 

In [ ]:
passive_reflection_operator = symmetric_machine.passive_reflection_operator
reflection_operator = symmetric_machine.even_machine_reflection_operator


def relative_reflection_error(values, parity=1):
    denominator = np.linalg.norm(values)
    if denominator == 0:
        return 0.0
    return np.linalg.norm(values - parity * values[:, ::-1]) / denominator


def relative_commutator_error(matrix):
    numerator = np.linalg.norm(
        reflection_operator @ matrix - matrix @ reflection_operator
    )
    return numerator / np.linalg.norm(matrix)


reflected_greens = np.einsum(
    "ij,jrz->irz", reflection_operator, eq._vgreen[:, :, ::-1]
)
resistance_matrix = np.diag(tokamak.coil_resist)
initial_currents = tokamak.getCurrentsVec()
symmetry_errors = {
    "sampled Green functions": (
        np.linalg.norm(eq._vgreen - reflected_greens)
        / np.linalg.norm(eq._vgreen)
    ),
    "inductance operator": relative_commutator_error(tokamak.coil_self_ind),
    "resistance operator": relative_commutator_error(resistance_matrix),
    "plasma flux": relative_reflection_error(eq.plasma_psi),
    "toroidal current density": relative_reflection_error(profiles.jtor),
    "metal currents": (
        np.linalg.norm(reflection_operator @ initial_currents - initial_currents)
        / np.linalg.norm(initial_currents)
    ),
}
limiter_defects = np.count_nonzero(
    eq.limiter_handler.mask_inside_limiter
    != eq.limiter_handler.mask_inside_limiter[:, ::-1]
)

for name, error in symmetry_errors.items():
    print(f"{name:27s}: {error:.3e}")
print(f"{'limiter-mask defects':27s}: {limiter_defects}")
assert max(symmetry_errors.values()) < 1e-12
assert limiter_defects == 0

### 3. Build the virtual circuits for the PCS

We use FreeGSNKE's `VirtualCircuitHandling` class (see Example 09) to build the virtual circuits for the five shaping circuits. It computes, by finite differences of static GS solves, the shape sensitivity matrix

$$
S_{kj} = \frac{\partial s_k}{\partial I_j},
$$

where $s_k$ are the shape targets (`Rin`, `Rout`, `Rx`, `Zx`) and $I_j$ the shaping-circuit currents, and its pseudo-inverse $S^+$ (the virtual circuits), which maps requested shape corrections to coil-current commands. The solenoid is excluded because it is reserved for plasma-current control.


In [ ]:
# functions for the plasma descriptors (shape functions)
def selected_lower_xpoint(equilibrium):
    """Select the lower null nearest the inverse-solve target."""
    lower = equilibrium.xpt[equilibrium.xpt[:, 1] < 0]
    target = np.array([0.8, -0.35])
    return lower[np.argmin(np.linalg.norm(lower[:, :2] - target, axis=1)), :2]


def plasma_descriptors(equilibrium):
    """Return the four shape measurements used by the controller."""
    rin, rout = equilibrium.innerOuterSeparatrix()
    rx, zx = selected_lower_xpoint(equilibrium)
    return np.array([rin, rout, rx, zx])

# calculate initial values
baseline_descriptors = plasma_descriptors(eq)

active_coils = list(tokamak.coils_list[: tokamak.n_active_coils])
shape_coils = [name for name in active_coils if name != "solenoid"]
ctrl_targets = ["Rin", "Rout", "Rx", "Zx"]

print("Active circuit order:", active_coils)
print("Shape parameters:", ctrl_targets)
print("Initial descriptors [m]:", baseline_descriptors)


In [ ]:
from freegsnke import virtual_circuits

# build the VCs
VCs = virtual_circuits.VirtualCircuitHandling()
VCs.define_solver(static_solver)
VCs.calculate_VC(
    eq=eq,
    profiles=profiles,
    coils=shape_coils,
    target_names=ctrl_targets,
    target_calculator=plasma_descriptors,
    n_vc_workers=2,
    verbose=True,
    name="VC_shape",
)

shape_matrix = VCs.VC_shape.shape_matrix  # [no. of targets x no. of coils], units m/A
shape_vcs = VCs.VC_shape.VCs_matrix  # [no. of coils x no. of targets], units A/m

### 4. Configure the plasma-control loop

Here we set up the FPDT controllers as in Example 11, holding the plasma current and all but one of the shape targets at their initial values, so that the controllers must counteract the resistive decay of the plasma current and any drift in the plasma shape. The X-point radius, $R_x$, is ramped outwards by 5 cm between 25% and 75% of the simulation. The shape controllers use proportional-integral (PI) control, so they remove steady offsets.

In [ ]:
from freegsnke.control_loop.pcs import PlasmaControlSystem

tmin = 0.0
tmax = 1.0
waveform_times = np.array([tmin, tmax])
zeros = {"times": waveform_times, "vals": np.zeros(2)}

plasma_data = {
    "ip_ref": {"times": waveform_times, "vals": np.full(2, profiles.Ip)},
    "vloop_ff": zeros,
    "ip_blend": {"times": waveform_times, "vals": np.ones(2)},
    "k_prop": {"times": np.array([tmin]), "vals": np.array([-20.0])},
    "k_int": {"times": np.array([tmin]), "vals": np.array([-100.0])},
    "k_deriv": {"times": np.array([tmin]), "vals": np.array([0.0])},
    "M_solenoid": {"times": np.array([tmin]), "vals": np.array([1.0])},
}

initial_shape = dict(zip(ctrl_targets, baseline_descriptors))
shape_data = {}
for target in ctrl_targets:
    shape_data[target] = {
        "ff": zeros,
        "ref": {
            "times": waveform_times,
            "vals": np.full(2, initial_shape[target]),
        },
        "blend": {"times": waveform_times, "vals": np.ones(2)},
        "k_prop": {"times": np.array([tmin]), "vals": np.array([25.0])},
        "k_int": {"times": np.array([tmin]), "vals": np.array([100.0])},
        "k_deriv": {"times": np.array([tmin]), "vals": np.array([0.0])},
    }

### Virtual circuits and electrical limits

The shape virtual circuits are the columns of $S^+$ (padded with zero on the solenoid), while the plasma-current virtual circuit acts solely on the solenoid. Electrical limits, resistances, and inductances are taken directly from the analytic tokamak object.

In [ ]:
n_active = tokamak.n_active_coils

circuits_data = {"coil_order": active_coils}
for target_index, target in enumerate(ctrl_targets):
    circuit = np.zeros(n_active)
    circuit[1:] = shape_vcs[:, target_index]
    circuits_data[target] = {"times": np.array([tmin]), "vals": [circuit]}
circuits_data["plasma"] = {
    "times": np.array([tmin]),
    "vals": [np.r_[1.0, np.zeros(n_active - 1)]],
}
for coil_name in active_coils:
    circuits_data[f"{coil_name}_ref"] = zeros

initial_active = tokamak.getCurrentsVec()[:n_active]
current_margin = np.maximum(5000.0, 2.0 * np.abs(initial_active))
min_current_lims = initial_active - current_margin
max_current_lims = initial_active + current_margin
# the solenoid current is limited to +/-10 kA
solenoid_index = active_coils.index("solenoid")
min_current_lims[solenoid_index] = -10e3
max_current_lims[solenoid_index] = 10e3
systems_data = {
    "min_coil_curr_lims": {
        "times": [tmin],
        "vals": [min_current_lims],
    },
    "max_coil_curr_lims": {
        "times": [tmin],
        "vals": [max_current_lims],
    },
    "max_coil_curr_ramp_lims": {
        "times": [tmin],
        "vals": [1e12 * np.ones(n_active)],
    },
}
for coil_name in active_coils:
    systems_data[f"{coil_name}_pert"] = zeros

pf_data = {
    "R_matrix": {"times": [tmin], "vals": [tokamak.coil_resist[:n_active]]},
    "M_FF_matrix": {
        "times": [tmin],
        "vals": [tokamak.coil_self_ind[:n_active, :n_active]],
    },
    "M_FB_matrix": {
        "times": [tmin],
        "vals": [tokamak.coil_self_ind[:n_active, :n_active]],
    },
    "coil_gains": {
        "times": [tmin],
        "vals": [0.015 * np.ones(n_active)],
    },
    "coil_voltage_lims": {
        "times": np.array([tmin]),
        "vals": [[2000.0] + [750.0] * (n_active - 1)],
    },
    "coil_voltage_slew_lims": {
        "times": [tmin],
        "vals": [1e12 * np.ones(n_active)],
    },
}
vertical_data = {
    "z_ref": {"times": waveform_times, "vals": np.zeros(2)},
    "k_prop": {"times": np.array([tmin]), "vals": np.array([0.0])},
    "k_deriv": {"times": np.array([tmin]), "vals": np.array([0.0])},
}
activation_data = {
    f"{name}_activation": {
        "times": np.array([tmin]),
        "vals": np.array([1.0]),
    }
    for name in active_coils
}

### Assemble the PCS

The standard `PCS` interface is unchanged: symmetry reduction is handled entirely by the evolutive solver.

In [ ]:
# build
pcs = PlasmaControlSystem(
    plasma_data=plasma_data,
    shape_data=shape_data,
    shape_control_mode="PID",
    circuits_data=circuits_data,
    systems_data=systems_data,
    pf_data=pf_data,
    vertical_data=vertical_data,
    coil_activation_data=activation_data,
    active_coils=active_coils,
    ctrl_coils=active_coils,
    vertical_coils=[],
    ctrl_targets=ctrl_targets,
    plasma_target=["plasma"],
)

In [ ]:
# ramp Rx outwards by 5 cm between 25% and 75% of the simulation (the other shape targets are held)
rx_initial = initial_shape["Rx"]
pcs.ShapeController.data["Rx"]["ref"] = {
    "times": np.array([tmin, 0.25 * tmax, 0.75 * tmax, tmax]),
    "vals": np.array([rx_initial, rx_initial, rx_initial + 0.05, rx_initial + 0.05]),
}
pcs.ShapeController.update_interpolants()


### 5. Initialise the reduced evolutive solver

In Example 10 we showed that the metals and the plasma are governed by the circuit equation (2) and the plasma equation (3). Linearising the plasma current density about the current equilibrium (equation (4)) gives the state-matrix system

$$
\mathbf{M}\dot{\mathbf{x}} + \mathbf{R}\mathbf{x} = \mathbf{c}, \qquad \mathbf{x} = \begin{bmatrix} \vec{I}_m \\ I_p \end{bmatrix}, \qquad A = -\mathbf{R}^{-1}\mathbf{M},
$$

where the eigenvalues of $A$ are the timescales $\tau_i$ of the system ($\tau_i > 0$ unstable, $\tau_i < 0$ stable). The `nl_solver` used in Examples 05 and 11 evolves all of these modes (or a subset if needed). This includes the fast, vertically unstable mode, whose short timescale forces a small timestep (0.5 ms in Example 11).

`SymmetryReducedSolver` removes that mode by exploiting the up-down symmetry of the machine and the equilibrium. Let $S_m$ be the reflection operator for the metal currents (it swaps each upper/lower partner) and $S_y$ the reflection of the plasma grid about $Z = 0$. For an exactly symmetric machine and equilibrium, reflected partners have equal resistances and inductances, and the Jacobians in equation (4) are even fields. The linearised system then commutes with reflection, $SA = AS$ with $S = \mathrm{diag}(S_m, 1)$, so the even and odd parts of the state evolve independently: an even perturbation never drives an odd response. The vertical instability is a vertical displacement of the plasma, which is odd, so it lives in the odd part and is removed by keeping only the even part.

#### How the passive modes are calculated

The passive structures obey the circuit equation (2) with no plasma and no applied voltage:

$$
M_{p,p}\dot{\vec{I}}_p + R_{p,p}\vec{I}_p = 0.
$$

We form $A_p = -R_{p,p}^{-1}M_{p,p}$ and solve the eigenproblem

$$
A_p\vec{v}_i = \tau_i\vec{v}_i.
$$

Each eigenvector $\vec{v}_i$ is a pattern of passive currents that decays as $e^{t/\tau_i}$. All $\tau_i$ are negative because the passive structures only dissipate energy. The modes are stored with their decay rates $1/|\tau_i|$ and sorted from lowest rate (slowest decay) upwards.

For $n=20$ passive elements, we obtain $n/2 = 10$ up-down symmetric pairs. The passive reflection operator $S_p$ is a permutation matrix, built from the pairing found by `prepare_up_down_symmetric_machine`, that swaps each upper/lower partner. For a pair $(u, l)$:

- the $S_p = +1$ eigenvector is $(e_u + e_l)/\sqrt{2}$: both partners carry the same current, giving flux that is **even** in $Z$.
- the $S_p = -1$ eigenvector is $(e_u - e_l)/\sqrt{2}$: the partners carry opposite currents, giving a field that is **odd** in $Z$. This is the kind of current that responds to vertical plasma motion.

The 10 pairs therefore give a 10-dimensional even subspace and a 10-dimensional odd subspace. A passive element on the midplane has no partner and contributes to the even subspace only.

We solve the eigenproblem separately in each subspace, so every mode has a definite parity, even when eigenvalues are degenerate. This requires $S_pA_p = A_pS_p$, which holds when the resistances and inductances are reflection-symmetric. The solver checks this and raises an error if it fails.

#### What the reduced solver evolves

We keep only (a subset of) the even modes. The metal currents are written as $\vec{I}_m = P\vec{\alpha}$, where the columns of $P$ are the active-coil unit vectors and the retained even passive modes (in the following cell, we will retain 6 of the 10 even modes). The odd modes are removed before the plasma coupling is built. The reduced state is then

$$
\mathbf{x}_+ = \begin{bmatrix} \vec{\alpha} \\ I_p \end{bmatrix} \qquad (6 \text{ active circuits} + 6 \text{ passive modes} + I_p = 13 \text{ values}).
$$

Substituting $\vec{I}_m = P\vec{\alpha}$ into equation (5), multiplying by $P^{-1}R_{m,m}^{-1}$, and dividing equation (6) by the lumped plasma resistance $R_p$ gives the same equations as Example 10 in the reduced state:

$$
\mathbf{M}_+\dot{\mathbf{x}}_+ + \mathbf{x}_+ = \mathbf{f},
$$

$$
\mathbf{M}_+ =
\begin{bmatrix}
P^{-1}R_{m,m}^{-1}\left( M_{m,m}P + M_{m,y}\dfrac{\partial \vec{I}_y}{\partial \vec{\alpha}} \right) &
P^{-1}R_{m,m}^{-1}M_{m,y}\dfrac{\partial \vec{I}_y}{\partial I_p} \\[2ex]
\dfrac{\hat{\vec{I}}_y^{T}}{R_p}\left( M_{y,m}P + M_{y,y}\dfrac{\partial \vec{I}_y}{\partial \vec{\alpha}} \right) &
\dfrac{\hat{\vec{I}}_y^{T}}{R_p}M_{y,y}\dfrac{\partial \vec{I}_y}{\partial I_p}
\end{bmatrix},
$$

$$
\mathbf{f} =
\begin{bmatrix} P^{-1}R_{m,m}^{-1}\vec{V}_m \\ 0 \end{bmatrix}
-
\begin{bmatrix} P^{-1}R_{m,m}^{-1}M_{m,y} \\[1ex] \dfrac{\hat{\vec{I}}_y^{T}}{R_p}M_{y,y} \end{bmatrix}
\dfrac{\partial \vec{I}_y}{\partial \vec{\theta}}\dot{\vec{\theta}}.
$$

The Jacobians $\partial\vec{I}_y/\partial\vec{\alpha}$, $\partial\vec{I}_y/\partial I_p$ and $\partial\vec{I}_y/\partial\vec{\theta}$ come from Grad-Shafranov solves forced to be even in $Z$, and are projected with $\Pi_+ = \tfrac{1}{2}(I + S_y)$ to remove odd round-off. The system is advanced with implicit Euler,

$$
\mathbf{x}_+(t + \Delta t) = (\mathbf{M}_+ + \Delta t\,\mathbf{I})^{-1}\left(\Delta t\,\mathbf{f} + \mathbf{M}_+\mathbf{x}_+(t)\right).
$$

After each step the equilibrium is updated with a Grad-Shafranov solve, and the Jacobians are rebuilt if the plasma current density has changed by more than the relinearisation threshold.

**Note:** Without the odd modes, the fastest remaining timescales are resistive, which allows a much larger timestep to be taken (recall the small timesteps used in Examples 05 and 11). This is exact only for the linearisation about an even equilibrium in an even machine: the plasma cannot move vertically, so it cannot be used for vertical displacement events or asymmetric scenarios. The timestep cannot be arbitrarily large, however: it must still resolve the retained dynamics, keep the linearisation accurate (the Jacobians are rebuilt whenever the plasma has changed significantly), and be small compared with the timescales and gains used in the virtual PCS.

In [ ]:
from freegsnke.symmetry_reduced_solve import SymmetryReducedSolver

# plasma resistivity [Ohm m]: sets how quickly the plasma current decays without control
plasma_resistivity = 1e-8

# initialise the solver object
stepping = SymmetryReducedSolver(
    eq=eq,
    profiles=profiles,
    GSStaticSolver=static_solver,
    full_timestep=2.5e-3,      # <-- 2.5 ms timestep!
    max_internal_timestep=2.5e-3,
    plasma_resistivity=plasma_resistivity,
    fix_n_vessel_modes=6,      # <-- 6/10 retained modes
    plasma_descriptor_function=plasma_descriptors,
    passive_reflection_operator=passive_reflection_operator,
    n_linearization_workers=2,
    verbose=False,
)


### Inspect passive-mode parity and truncation

Open markers show the discarded passive modes (even in blue, odd in orange); filled markers show the even modes retained after parity and mode-count selection (we keep the 6 slowest-decaying even modes). The spatial panels show representative even and odd mode current patterns.

In [ ]:
normal_modes = stepping.evol_metal_curr.normal_modes
passive_parity = normal_modes.passive_mode_parity
passive_frequency = normal_modes.w_passive
retained_passive = stepping.evol_metal_curr.selected_modes_mask[n_active:]
passive_centres = np.array(
    [
        [np.mean(element["R"]), np.mean(element["Z"])]
        for element in symmetric_machine.passive_coils_data
    ]
)

representative_modes = {
    "even mode": np.flatnonzero(passive_parity > 0)[0],
    "odd mode": np.flatnonzero(passive_parity < 0)[0],
}
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for parity, label, colour in ((1, "even", "tab:blue"), (-1, "odd", "tab:orange")):
    parity_mask = (passive_parity == parity) & ~retained_passive
    axes[0].scatter(
        np.flatnonzero(parity_mask),
        passive_frequency[parity_mask],
        facecolors="none",
        edgecolors=colour,
        label=f"{label}, discarded",
    )
retained_indices = np.flatnonzero(retained_passive)
axes[0].scatter(
    retained_indices,
    passive_frequency[retained_passive],
    color="tab:blue",
    label="even, retained",
)
axes[0].set_yscale("log")
axes[0].set_xlabel("passive-mode index")
axes[0].set_ylabel("frequency [s$^{-1}$]")
axes[0].set_title("Parity-aware mode selection")
axes[0].grid(True)
axes[0].legend()

for axis, (title, mode_index) in zip(axes[1:], representative_modes.items()):
    amplitudes = normal_modes.Pmatrix[
        n_active:, n_active + mode_index
    ]
    scale = np.max(np.abs(amplitudes))
    points = axis.scatter(
        passive_centres[:, 0],
        passive_centres[:, 1],
        c=amplitudes,
        cmap="coolwarm",
        vmin=-scale,
        vmax=scale,
        s=70,
        edgecolor="k",
    )
    axis.axhline(0.0, color="0.6", linewidth=1)
    axis.set_aspect("equal")
    axis.set_xlabel("R [m]")
    axis.set_ylabel("Z [m]")
    axis.set_title(title)
    fig.colorbar(points, ax=axis, label="relative mode current")

fig.tight_layout()
plt.show()

### 6. Run the controlled evolution

Next, we'll run the simulation with a 2.5 ms solver step and a 1.25 ms PCS timestep.

The virtual PCS will return the voltages to be applied to the active coils, which the reduced solver will use to advance the retained metal modes, plasma current, and profile response (and equilibrium!).

The PCS holds the plasma current at its initial value, and all shape targets at their initial values except $R_x$, which is ramped outwards by 5 cm. The plasma resistivity is set to $10^{-8}\,\Omega\,\mathrm{m}$, so the resistive loss in plasma current that the solenoid must make up stays within its $\pm 10$ kA current limit.

In [ ]:
# setup and storage
dt = stepping.dt_step
n_steps = int(round((tmax - tmin) / dt)) + 1  # simulate from tmin to tmax
dt_pcs = dt / 2
times = tmin + np.arange(n_steps) * dt
stepping.initialize_from_ICs(eq, profiles)

# keep four equilibrium snapshots, evenly spaced in time, for plotting later
snapshot_indices = np.linspace(0, n_steps - 1, 4).astype(int)
snapshots = {0: stepping.eq1.create_auxiliary_equilibrium()}

# re-linearisation: the solver re-linearises when the relative change in the toroidal current density
# since the last linearisation reaches this threshold (we store this criterion at every step)
relinearise_threshold = 0.05
relinearise_criterion = np.zeros(n_steps - 1)

dynamic_psi = np.zeros((*eq.psi().shape, n_steps))
dynamic_currents = np.zeros((len(stepping.vessel_currents_vec), n_steps))
dynamic_ip = np.zeros(n_steps)
dynamic_shape = np.zeros((len(ctrl_targets), n_steps))
dynamic_z = np.zeros(n_steps)
voltages = np.zeros((n_active, n_steps - 1))
step_times = np.zeros(n_steps - 1)

dynamic_psi[:, :, 0] = stepping.eq1.psi()
dynamic_currents[:, 0] = stepping.vessel_currents_vec
dynamic_ip[0] = stepping.profiles1.Ip
dynamic_shape[:, 0] = plasma_descriptors(stepping.eq1)
dynamic_z[0] = stepping.eq1.Zcurrent()

ip_hist_prev = 0.0
ip_err_prev = 0.0
shape_error_prev = np.zeros(len(ctrl_targets))
shape_history_prev = np.zeros(len(ctrl_targets))
approved_current_prev = initial_active.copy()
approved_voltage_prev = np.zeros(n_active)

In [ ]:
# simulate!
for index, time_value in enumerate(times[:-1]):
    
    print("-----")
    print(f"t = {np.round(time_value,5)}s (step {index+1}/{n_steps-1})")
    
    
    result = pcs.calculate_ctrl_voltages(
        t=time_value,
        dt=dt_pcs,
        dt_simulator=dt,
        ip_meas=dynamic_ip[index],
        ip_hist_prev=ip_hist_prev,
        ip_err_prev=ip_err_prev,
        T_meas=dynamic_shape[:, index],
        T_err_prev=shape_error_prev,
        T_hist_prev=shape_history_prev,
        I_approved_prev=approved_current_prev,
        I_meas=dynamic_currents[:n_active, index],
        V_approved_prev=approved_voltage_prev,
        zip_meas=dynamic_z[index] * dynamic_ip[index],
        zipv_meas=0.0,
        active_coil_resists=tokamak.coil_resist[:n_active],
        verbose=False,
    )
    (
        approved_voltage,
        ip_hist_prev,
        ip_err_prev,
        shape_error_prev,
        shape_history_prev,
        approved_current_prev,
        active_resistances,
    ) = result
    voltages[:, index] = approved_voltage
    approved_voltage_prev = approved_voltage.copy()

    start = time.perf_counter()
    stepping.nlstepper(
        plasma_resistivity=plasma_resistivity,
        active_voltage_vec=approved_voltage,
        # Lao logic adds one dependent closure coefficient to each stored
        # array; the evolutive interface accepts the independent values only.
        profiles_parameters={
            "alpha": profiles.alpha[:-1].copy(),
            "beta": profiles.beta[:-1].copy(),
        },
        custom_active_coil_resistances=active_resistances,
        linear_only=True,
        target_relative_tol_currents=1e-2,
        target_relative_tol_GS=1e-2,
        working_relative_tol_GS=5e-3,
        max_solving_iterations=20,
        relinearise_threshold=relinearise_threshold,
        no_GS=False,
        verbose=False,
    )
    step_times[index] = time.perf_counter() - start
    relinearise_criterion[index] = stepping.relinearise_criteria

    # dynamic_psi[:, :, index + 1] = stepping.eq1.psi()
    dynamic_currents[:, index + 1] = stepping.vessel_currents_vec
    dynamic_ip[index + 1] = stepping.profiles1.Ip
    dynamic_shape[:, index + 1] = plasma_descriptors(stepping.eq1)
    dynamic_z[index + 1] = stepping.eq1.Zcurrent()
    if index + 1 in snapshot_indices:
        snapshots[index + 1] = stepping.eq1.create_auxiliary_equilibrium()


    # print some stuff to track solve
    print(f"      PLASMA QUANTITIES")
    print(f"        Ip = {np.round(dynamic_ip[index+1]/1000,1)} [kA]")
    print(f"        Shape parameters {ctrl_targets} = {np.round(dynamic_shape[:,index+1],3)}")
    print(f"        Z position = {np.round(dynamic_z[index+1],5)} [m]")


### 7. Inspect the results

Below we plot the plasma current and shape against their reference targets, followed by the active coil currents (with the solenoid's PCS current limits) and voltages, snapshots of the equilibrium as it evolves, and finally the passive structure currents.

Note the slightly non-physical solenoid current that is compensating a bit for the lack of shaping coils. 

In [ ]:
t_ms = 1e3 * times
shape_reference = np.array(  # reference waveform of each shape target, evaluated at the simulation times
    [
        np.interp(
            times,
            pcs.ShapeController.data[target]["ref"]["times"],
            pcs.ShapeController.data[target]["ref"]["vals"],
        )
        for target in ctrl_targets
    ]
)
t_voltage_ms = 1e3 * times[:-1]  # each voltage request is applied from the start of its step
colours = {name: f"C{index}" for index, name in enumerate(active_coils)}

# plasma current and shape: evolved values against their reference targets
ip_reference = np.interp(times, plasma_data["ip_ref"]["times"], plasma_data["ip_ref"]["vals"])
fig, axes = plt.subplots(2, 3, figsize=(12, 6), dpi=80, sharex=True, constrained_layout=True)
ip_axis = axes[0, 0]
shape_axes = [axes[0, 1], axes[0, 2], axes[1, 0], axes[1, 1]]
axes[1, 2].axis("off")  # unused

ip_axis.plot(t_ms, dynamic_ip / 1e3, color="k", label="evolved")
ip_axis.plot(t_ms, ip_reference / 1e3, color="tab:red", linestyle="--", label="reference")
ip_axis.set_ylabel(r"Plasma current, $I_p$ [kA]")
ip_axis.legend()

for target_index, (axis, target) in enumerate(zip(shape_axes, ctrl_targets)):
    axis.plot(t_ms, dynamic_shape[target_index], color="k", label="evolved")
    axis.plot(t_ms, shape_reference[target_index], color="tab:red", linestyle="--", label="reference")
    axis.set_ylabel(f"{target} [m]")

for axis in [ip_axis] + shape_axes:
    axis.grid(alpha=0.5)
for axis in (axes[0, 2], axes[1, 0], axes[1, 1]):
    axis.tick_params(labelbottom=True)
    axis.set_xlabel("Time [ms]")
fig.suptitle("Plasma current and shape: evolved against reference")
plt.show()

# active coil currents and voltages (solenoid and shape coils are shown separately)
solenoid = active_coils.index("solenoid")
shape_coil_indices = [index for index in range(n_active) if index != solenoid]
min_current_limits = systems_data["min_coil_curr_lims"]["vals"][0]
max_current_limits = systems_data["max_coil_curr_lims"]["vals"][0]

fig, axes = plt.subplots(2, 2, figsize=(12, 6), dpi=80, sharex=True, constrained_layout=True)

axes[0, 0].plot(t_ms, dynamic_currents[solenoid] / 1e3, color=colours["solenoid"], label="solenoid")
axes[0, 0].axhline(max_current_limits[solenoid] / 1e3, color="tab:red", linestyle="--", linewidth=1, label="PCS limits")
axes[0, 0].axhline(min_current_limits[solenoid] / 1e3, color="tab:red", linestyle="--", linewidth=1)
axes[0, 0].set_ylabel("Solenoid current [kA]")
axes[0, 0].legend()

axes[1, 0].step(t_voltage_ms, voltages[solenoid], where="post", color=colours["solenoid"])
axes[1, 0].set_ylabel("Solenoid voltage [V]")

for index in shape_coil_indices:
    name = active_coils[index]
    axes[0, 1].plot(t_ms, dynamic_currents[index] - dynamic_currents[index, 0], color=colours[name], label=name)
    axes[1, 1].step(t_voltage_ms, voltages[index], where="post", color=colours[name], label=name)
axes[0, 1].set_ylabel("Shape-coil current change [A]")
axes[1, 1].set_ylabel("Shape-coil voltages [V]")
axes[0, 1].legend(ncol=2)

for axis in axes.ravel():
    axis.grid(alpha=0.5)
for axis in axes[1]:
    axis.set_xlabel("Time [ms]")
fig.suptitle("Active coil currents and voltages")
plt.show()


In [ ]:
from matplotlib.lines import Line2D

# evolution of the equilibrium: four snapshots, evenly spaced in time
fig, axes = plt.subplots(1, 4, figsize=(15, 4.5), dpi=80, sharex=True, sharey=True, constrained_layout=True)
initial = snapshots[0]
for axis, index in zip(axes, snapshot_indices):
    snapshot = snapshots[index]
    snapshot.tokamak.plot(axis=axis, show=False)  # plots the active coils and passive structures
    snapshot.plot(axis=axis, show=False)  # plots the equilibrium and its separatrix
    axis.set_title(f"t = {1e3 * times[index]:.0f} ms")
    axis.grid(alpha=0.5)
for axis in axes[1:]:
    axis.set_ylabel("")
axes[0].set_xlim(eq.Rmin, eq.Rmax)  # show the full computational domain, including the coils
axes[0].set_ylim(eq.Zmin, eq.Zmax)
axes[-1].legend(
    handles=[
        Line2D([], [], color="r", label="separatrix"),
        Line2D([], [], color="r", marker="x", linestyle="", label="X-points"),
        Line2D([], [], color="g", marker="2", linestyle="", label="O-point"),
    ],
    loc="lower right",
    fontsize="small",
)

plt.show()


In [ ]:
# passive structure currents (the passive elements follow the active coils in the stored current vector)
passive_currents = dynamic_currents[n_active:]
passive_names = symmetric_machine.passive_names
passive_centres = np.array([[np.mean(element["R"]), np.mean(element["Z"])] for element in symmetric_machine.passive_coils_data])
scale, unit = (1e-3, "kA") if np.max(np.abs(passive_currents)) > 1e3 else (1.0, "A")

fig, (time_axis, space_axis) = plt.subplots(1, 2, figsize=(13, 5), dpi=80, constrained_layout=True)

# time evolution: reflected partners carry identical currents, so only the upper elements are drawn
upper = [index for index, name in enumerate(passive_names) if name.endswith("_upper")]
for colour, index in zip(plt.cm.viridis(np.linspace(0, 1, len(upper))), upper):
    time_axis.plot(t_ms, scale * passive_currents[index], color=colour, label=passive_names[index].replace("_upper", ""))
time_axis.set_xlabel("Time [ms]")
time_axis.set_ylabel(f"Passive structure current [{unit}]")
time_axis.set_title("Passive currents (upper elements; the lower partners are identical)")
time_axis.legend(ncol=2, fontsize="small")
time_axis.grid(alpha=0.5)

# spatial distribution at the end of the simulation
limit = scale * np.max(np.abs(passive_currents[:, -1]))
points = space_axis.scatter(
    passive_centres[:, 0],
    passive_centres[:, 1],
    c=scale * passive_currents[:, -1],
    cmap="coolwarm",
    vmin=-limit,
    vmax=limit,
    s=120,
    edgecolor="k",
    zorder=3,
)
space_axis.plot(tokamak.limiter.R, tokamak.limiter.Z, "k-", linewidth=1.2)
space_axis.set_aspect("equal")
space_axis.set_xlabel("Major radius [m]")
space_axis.set_ylabel("Height [m]")
space_axis.set_title(f"Passive currents at t = {t_ms[-1]:.0f} ms")
space_axis.grid(alpha=0.5)
fig.colorbar(points, ax=space_axis, label=f"Passive structure current [{unit}]")
plt.show()

upper_lower_difference = max(
    np.max(np.abs(passive_currents[index] - passive_currents[passive_names.index(passive_names[index].replace("_upper", "_lower"))]))
    for index in upper
)
print(f"Maximum upper/lower passive current difference: {upper_lower_difference:.2e} A")


The passive structure currents are even in $Z$: each upper element carries exactly the same current as its lower partner. The largest changes occur when the $R_x$ ramp starts and ends. 


Finally, we look at the wall-clock time taken by each step of the evolutive solver (this excludes the virtual PCS calculation). The red markers show the steps in which the solver re-linearised, i.e. where the relative change in the toroidal current density since the last linearisation reached the threshold (5%). Re-linearisations are the most expensive steps, as the Jacobians are rebuilt.


In [ ]:
# wall-clock time of each evolutive solver step (the virtual PCS calculation is not included)
median_step = np.median(step_times)
t_step_ms = 1e3 * times[1:]  # time at the end of each step
relin_steps = np.flatnonzero(relinearise_criterion >= relinearise_threshold)  # steps in which the solver re-linearised

fig, (step_axis, total_axis) = plt.subplots(1, 2, figsize=(13, 4.5), dpi=80, constrained_layout=True)

step_axis.semilogy(t_step_ms, step_times, color="k", linewidth=0.8, label="all steps")
step_axis.semilogy(t_step_ms[relin_steps], step_times[relin_steps], "v", color="tab:red", markersize=7, label=f"re-linearisations ({len(relin_steps)})")
step_axis.axhline(median_step, color="tab:blue", linestyle="--", linewidth=1, label=f"median step ({1e3 * median_step:.0f} ms)")
step_axis.set_xlabel("Time [ms]")
step_axis.set_ylabel("Wall-clock time per solver step [s]")
step_axis.set_title("Time per step")
step_axis.legend(loc="lower right")
step_axis.grid(alpha=0.5, which="both")

total_axis.plot(t_step_ms, np.cumsum(step_times), color="k")
for step in relin_steps:
    total_axis.axvline(t_step_ms[step], color="tab:red", linestyle=":", linewidth=1)
total_axis.plot([], [], color="tab:red", linestyle=":", label="re-linearisation")
total_axis.set_xlabel("Simulated time [ms]")
total_axis.set_ylabel("Cumulative wall-clock time [s]")
total_axis.set_title(f"Total: {np.sum(step_times):.0f} s for {t_step_ms[-1]:.0f} ms of plasma evolution")
total_axis.legend()
total_axis.grid(alpha=0.5)
plt.show()

print(f"{len(step_times)} steps of {1e3 * dt:.1f} ms | median {1e3 * median_step:.0f} ms | total {np.sum(step_times):.0f} s")
print(f"{len(relin_steps)} re-linearisations, at t = {np.round(1e3 * times[relin_steps], 1)} ms")
